# Choosing a Training Method — a Use-Case Lab

A **conceptual companion** to Parts 1–4. No model training here — the goal is judgment: given a real situation, *which* adaptation method do you reach for, and why? You've now built the mechanics of continued pretraining, full fine-tuning, a frozen-encoder probe, FinBERT, and QLoRA. This notebook is about knowing **when** to use each.

- **Part A — Triage deck:** 8 scenarios. Commit to a method + a one-line justification, then check the answer-key notebook.
- **Part B — Predict-the-ranking:** predict how your own methods rank on accuracy *and* cost, then fill the table and reconcile.

*(This is the `PRACTICE` file if the title says so — write your answers here, then open `Choosing_a_Training_Method.ipynb` to compare.)*

## The framework: what are you adapting, and what does it cost?

Every method changes a **different amount** of the model. From cheapest to most invasive:

| Method | What changes | Labeled data | Compute | Your example |
|---|---|---|---|---|
| **Prompting / zero-shot** | nothing | none | inference only | FinBERT's own head (P3 §8a) |
| **Freeze backbone, train head** | a tiny linear layer | very little | cheap | P3 §8b probe |
| **Full fine-tune** | every weight | moderate–lots | expensive | Parts 2 & 3 |
| **LoRA / QLoRA** | small adapters (~1%) | moderate | cheap-ish, fits big models | Part 4 |
| **Continued pretraining** | the backbone's *domain sense*, from **unlabeled** text | none (unlabeled) | expensive, but reusable | Part 1 (FinBERT productizes it) |
| **Pretrain from scratch** | everything, from zero | none (unlabeled, huge) | enormous | (almost never) |

### The five decision drivers
Whenever you choose, weigh these five:
1. **Labeled data** — how many *labeled* examples? (Little ⇒ freeze/prompt; lots ⇒ full fine-tune.)
2. **Domain distance** — how far is your text from the base model's pretraining? (Far ⇒ pretraining / a domain model helps.)
3. **Compute & memory** — what hardware/budget? (Tight ⇒ freeze or QLoRA, not full FT of a big model.)
4. **Number of downstream tasks** — one, or many on the same domain? (Many ⇒ pretraining / shared frozen backbone amortizes.)
5. **Serving constraints** — latency, cost, maintainability in production? (Strict ⇒ small/shared models.)

**The meta-rule:** pick the *least-effort* method that clears your accuracy bar; escalate only when it doesn't.

## Part A — Scenario triage  &larr; **your judgment**

For each scenario, fill in the `✍️ Your call` cell: the **method**, the **driver(s)** that tipped it, a one-sentence **why**, and **what would change your answer**. Don't aim for a single 'right' token — the reasoning is the point. Then compare with the answer key.

*(There isn't always one correct method; several scenarios are deliberately a judgment call between two.)*

### Scenario 1: FinBERT-style investment

A fintech has **5 billion words** of unlabeled analyst reports and filings, but only **~500 labeled** sentiment sentences. They plan to build **several** classifiers on this domain (sentiment, risk-flagging, topic). Generic LLMs handle finance jargon poorly. A multi-GPU budget is available.

#### ✅ Recommended — Scenario 1
- **Method:** **Continued pretraining** on the unlabeled corpus → then fine-tune / probe each task.
- **Drivers:** domain distance (far), *abundant unlabeled* text, *many* downstream tasks, scarce labels.
- **Why:** a one-time domain-adaptation of the backbone makes every later task start ahead — this is literally the FinBERT recipe, and the cost amortizes over the multiple planned tasks.
- **What would change it:** if a finance model (FinBERT/FinMA) already fits, *use it instead* of pretraining yourself; if there were only **one** task, the ROI would mostly vanish (see Scenario 7).

### Scenario 2: Tiny labeled set, ships next week

A startup has **200 labeled** examples for **one** classification task, a **strong general** base model, and a **single modest GPU**. Accuracy needs to be 'decent,' and it ships in a week.

#### ✅ Recommended — Scenario 2
- **Method:** **Freeze the backbone, train a linear head** (and/or few-shot prompt as a baseline).
- **Drivers:** very little labeled data (full FT would overfit), low compute, single task, deadline.
- **Why:** a probe learns just a small head, so it's fast and robust on tiny data, and the strong base's features are likely already good enough for 'decent.'
- **What would change it:** more labeled data (→ full fine-tune); a far-from-base domain (→ a domain model or LoRA).

### Scenario 3: Plenty of data, close domain

**80,000 labeled** product reviews, **one** sentiment task. The general base model is **already strong** on consumer text. GPUs are available, and you want the **best** achievable accuracy.

#### ✅ Recommended — Scenario 3
- **Method:** **Full fine-tuning.**
- **Drivers:** ample labels (no overfitting risk), domain *close* to base, compute available, accuracy is the priority.
- **Why:** with lots of in-distribution labels, letting the whole backbone adapt gives the highest ceiling, and there's plenty of data to support it.
- **What would change it:** if the domain were *far* from base, you'd add pretraining / a domain model first; a frozen probe here would leave accuracy on the table; **pretraining would be wasted** (domain is already close).

### Scenario 4: Big model, small GPU

You want the reasoning quality of a **13B** LLM for a classification/extraction task, you have **~10,000 labeled** examples, but only a **single 16 GB GPU**.

#### ✅ Recommended — Scenario 4
- **Method:** **QLoRA / PEFT.**
- **Drivers:** model too big to full-fine-tune on this hardware, but you *do* want to adapt it; moderate labels justify training.
- **Why:** 4-bit weights + small LoRA adapters make a 13B model trainable on one GPU — full FT is simply impossible here, and a frozen probe would underuse the model.
- **What would change it:** a tiny model that fits full FT (→ just full FT); near-zero labels (→ prompt the 13B zero/few-shot instead).

### Scenario 5: Need it today, no labels

A PM asks for a rough sentiment tagger **by tomorrow**. There's **no labeled data**, no training pipeline, but you have API access to a **capable general LLM**.

#### ✅ Recommended — Scenario 5
- **Method:** **Prompting** — zero-shot first, few-shot if accuracy falls short (your Part 5).
- **Drivers:** no labeled data, no training time/budget, a capable base is available.
- **Why:** it's the only method that needs *zero* training — the no-training baseline that gets you a number today.
- **What would change it:** once you can collect labels and care about cost/latency at scale, **distill** the prompt's outputs into a small fine-tuned model.

### Scenario 6: Many tasks, one backbone, low latency

A platform must run **12 different** text classifiers (sentiment, topic, toxicity, language, …) **cheaply** and with **low latency**. Maintaining 12 separately fine-tuned full models is too costly.

#### ✅ Recommended — Scenario 6
- **Method:** **Frozen shared backbone + 12 small heads** (or per-task LoRA adapters).
- **Drivers:** many tasks, serving cost & latency, maintainability.
- **Why:** one backbone stays in memory and you swap cheap per-task heads — huge serving savings for a small per-task accuracy hit.
- **What would change it:** if a couple of tasks need higher accuracy, give *those* per-task LoRA adapters (a richer middle ground than a bare linear head).

### Scenario 7: Tempted to pretrain — but should you?

You have a **large unlabeled finance corpus** and **one** upcoming sentiment task. You're tempted to continue-pretrain GPT-2 for **weeks**. The deadline is tight — and **FinBERT already exists**.

#### ✅ Recommended — Scenario 7
- **Method:** **Don't pretrain yourself** — use **FinBERT** (the pretraining is already done) and fine-tune it, or just fine-tune a good base.
- **Drivers:** single task (no amortization), tight time, an existing domain model.
- **Why:** continued-pretraining ROI needs *multiple* tasks **or** the absence of a domain model; with one task and FinBERT on the shelf, reinventing it is wasted effort. (This is the 'when **not** to pretrain' lesson — and exactly why your Part 1→3 arc lands on FinBERT.)
- **What would change it:** several planned tasks **and** no suitable domain model (→ Scenario 1).

### Scenario 8: No suitable base model exists

A genomics lab has **billions of DNA-sequence tokens**, a **unique vocabulary** unlike any pretrained model, a **large compute grant**, and **many** planned downstream predictors.

#### ✅ Recommended — Scenario 8
- **Method:** **Pretrain from scratch** (or heavy continued pretraining on the closest available base).
- **Drivers:** *no suitable base exists* (novel domain & vocabulary), abundant unlabeled data, big budget, many downstream tasks.
- **Why:** when nothing was pretrained on anything like your data, there's no backbone to adapt — you have to build the representations. This is the **rare** case that justifies from-scratch.
- **What would change it:** the existence of *any* reasonable base (→ continued pretraining instead, far cheaper). For almost everyone in almost every domain: **don't** pretrain from scratch.

### Part A wrap-up

Notice the recurring tells:
- **Few labels** → freeze / prompt. **Many labels** → full fine-tune.
- **Far domain + unlabeled text + many tasks** → pretraining (or grab a domain model). **One task or close domain** → don't.
- **Big model, small GPU** → QLoRA. **Many tasks, tight serving** → shared frozen backbone.
- **No time / no labels** → prompt, then distill later.

Almost every real decision is just these five drivers pulling against each other.

## Part B — Predict the ranking  &larr; **your judgment, then your data**

You've now run (or will run) the same Financial PhraseBank task with up to seven methods. Before looking at the numbers, **commit to two predictions**. Then fill the table and see where reality surprised you.

#### ✅ A reasonable prediction
**Accuracy, best → worst (full-fine-tune family clustered high):**
1. FinBERT zero-shot *(but inflated — it trained on PhraseBank; not a fair bar)*
2. FinBERT fine-tuned
3. BERT-base fine-tuned
4. Mistral-7B QLoRA *(strong, but not guaranteed to top a well-matched FinBERT here)*
5. GPT-2 fine-tuned (EDGAR-pretrained)
6. GPT-2 fine-tuned (base)
7. BERT-base frozen probe *(raw features only — expect the biggest drop)*

**Training cost, cheapest → most expensive:**
1. FinBERT zero-shot (no training)  2. Frozen probe (~2k params)  3–5. GPT-2 / BERT / FinBERT full FT (100M+ params each)  6. QLoRA Mistral (adapters are cheap to train, but it's a 7B model — memory-heavy)  7. Continued pretraining (Part 1) as a *separate* up-front cost.

**Biggest expected surprise:** the top of the accuracy table is *compressed* (mid-90s) — a cheap FinBERT is about as good as an expensive QLoRA-7B here, i.e. **diminishing returns**.

### The results table
Fill `test_acc` from **your** runs (leave `None` for parts you haven't run yet), and assign each method a rough `train_cost` on a **0–5** ordinal (0 = no training, 5 = most expensive). The cell sorts it two ways so the ranking falls out.

In [ ]:
import pandas as pd

# test_acc: filled with the numbers you've already produced; None = run it and fill in.
# train_cost: a rough 0-5 ordinal (0 = no training at all, 5 = most expensive).
rows = [
    # method,                              part,     test_acc, train_cost, notes
    ('GPT-2 fine-tuned (base)',            'P2',      82.1,    3, 'full FT, 124M params'),
    ('GPT-2 fine-tuned (EDGAR-pretrn)',    'P2',      86.5,    3, 'full FT; pretraining was a separate up-front cost'),
    ('BERT-base [CLS] fine-tuned',         'P3',      95.1,    3, 'full FT, 110M params'),
    ('FinBERT [CLS] fine-tuned',           'P3',      97.6,    3, 'full FT on a domain-pretrained backbone'),
    ('FinBERT zero-shot (own head)',       'P3 8a',   None,    0, 'NO training; but leakage: trained on PhraseBank'),
    ('BERT-base frozen probe',             'P3 8b',   None,    1, 'only the ~2k-param head trains'),
    ('Mistral-7B QLoRA',                   'P4',      None,    2, 'adapters cheap to train, but a 7B model: memory-heavy'),
]
df = pd.DataFrame(rows, columns=['method', 'part', 'test_acc', 'train_cost', 'notes'])

print('--- by ACCURACY (best first) ---')
print(df.sort_values('test_acc', ascending=False, na_position='last').to_string(index=False))
print('\n--- by TRAINING COST (cheapest first) ---')
print(df.sort_values('train_cost', ascending=True).to_string(index=False))

### Reconcile — what the numbers teach

- **The top is compressed.** Mid-90s for BERT, FinBERT, and (likely) QLoRA. The expensive 7B model doesn't dominate a well-matched FinBERT on a task this clean — **diminishing returns** once you're near the ceiling.
- **The frozen probe is the cheap floor.** Expect it well below the ~95% full fine-tune; that gap **is** the value of letting the backbone adapt. (Cheap, but a real ceiling.)
- **Zero-shot FinBERT looks unbeatable — and isn't a fair bar.** It was trained on PhraseBank, so its score is leakage-inflated. A reminder to always ask *what did this model already see?*
- **Cost and accuracy don't align.** The cheapest decent option (a probe, or an off-the-shelf domain model) may already clear your bar; the priciest (QLoRA-7B) may not justify itself **here**.

**The takeaway, restated:** start from the cheapest method that could plausibly clear your accuracy bar, and only climb the cost ladder when the numbers force you to.